# v52 — KTF³ vs Auger UHECR: Exposure-Corrected Dipole Test

**KTF³ Analysis — Andrew Cotting, April 2026**

## Hypothesis

The KTF³ preferred axis (l=277°, b=−31°) should align with the cosmic-ray arrival dipole  
if UHECR sources trace the large-scale structure defined by Klein bottle topology.

## Data

**Pierre Auger Observatory Open Data** — https://opendata.auger.org  
Surface Detector catalog (E > 32 EeV, ~109 events, 2004–2020).  
File: `auger_catalogSD.csv` (upload from Auger open data portal)

## Method

1. Compute measured CR dipole direction from data
2. Correct for Auger exposure (observatory at lat = −35.2°, southern sky bias)
3. Compare corrected dipole to KTF³ axis → angular separation
4. Monte Carlo null: exposure-weighted random isotropic sky

## Known results

- v52 (naive, no exposure correction): σ = 6.31 → **ARTEFACT** (southern sky bias)
- v52b (exposure-corrected): σ = 0.42, separation = 22.9° → **NULL**
- This notebook (v52 clean): same as v52b, improved code and documentation

## Verdict

NULL. The 22.9° geometric alignment between KTF³ axis and Auger dipole literature value  
is consistent with chance given the exposure-corrected null distribution.  
This result does not falsify KTF³ (insufficient statistics: N=109 events).

In [ ]:
!pip install numpy scipy matplotlib astropy healpy pandas -q
print('Packages ready.')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy import stats
import healpy as hp
import astropy.units as u
from astropy.coordinates import SkyCoord
import os, warnings
warnings.filterwarnings('ignore')

# KTF3 axis
KTF3_L = 277.0
KTF3_B = -31.0

# Auger observatory location (Malargüe, Argentina)
AUGER_LAT = -35.2  # degrees

# Literature: Auger 2017 dipole direction (Aab+2017, Science)
AUGER_LIT_L = 233.0  # galactic longitude
AUGER_LIT_B =  -13.0  # galactic latitude

# Convert KTF3 axis
ktf3_coord = SkyCoord(l=KTF3_L*u.deg, b=KTF3_B*u.deg, frame='galactic')
print(f'KTF³ axis: l={KTF3_L}°, b={KTF3_B}°')
print(f'Auger lat: {AUGER_LAT}°')

def gal_to_vec(l_deg, b_deg):
    l, b = np.radians(l_deg), np.radians(b_deg)
    return np.array([np.cos(b)*np.cos(l), np.cos(b)*np.sin(l), np.sin(b)])

ktf3_vec = gal_to_vec(KTF3_L, KTF3_B)
auger_lit_vec = gal_to_vec(AUGER_LIT_L, AUGER_LIT_B)
sep_lit = np.degrees(np.arccos(np.clip(np.dot(ktf3_vec, auger_lit_vec), -1, 1)))
print(f'Separation KTF³ vs Auger literature dipole: {sep_lit:.1f}°')

In [ ]:
# === LOAD AUGER DATA ===
# Download from: https://opendata.auger.org → UHECR Catalog → Surface Detector
# File: auger_catalogSD.csv
# Upload to Colab via Files panel (left sidebar)

# Auto-rename if file has spaces/parentheses from browser download
for f in os.listdir('.'):
    if 'auger' in f.lower() and f.endswith('.csv') and f != 'auger_catalogSD.csv':
        os.rename(f, 'auger_catalogSD.csv')
        print(f'Renamed: {f} → auger_catalogSD.csv')

USE_SYNTHETIC = False
if os.path.exists('auger_catalogSD.csv'):
    df = pd.read_csv('auger_catalogSD.csv')
    print(f'Auger data loaded: {len(df)} events')
    print(df.head())
    print(f'Columns: {list(df.columns)}')
else:
    print('⚠️  auger_catalogSD.csv not found — using SYNTHETIC data.')
    print('    Download from: https://opendata.auger.org')
    print('    Upload to Colab before running this cell.')
    USE_SYNTHETIC = True
    np.random.seed(42)
    N = 109
    # Synthetic: isotropic above Auger horizon (dec > -90, limited by exposure)
    ra_syn  = np.random.uniform(0, 360, N)
    dec_syn = np.degrees(np.arcsin(np.random.uniform(
        np.sin(np.radians(AUGER_LAT - 60)),
        np.sin(np.radians(AUGER_LAT + 60)), N)))
    energy_syn = np.random.uniform(32, 120, N)
    df = pd.DataFrame({'RA': ra_syn, 'Dec': dec_syn, 'Energy': energy_syn})

In [ ]:
# === PARSE COORDINATES ===
# Adapt column names to actual CSV structure
cols = [c.lower() for c in df.columns]
col_map = {}
for c in df.columns:
    cl = c.lower()
    if 'ra' in cl or 'alpha' in cl: col_map['ra'] = c
    if 'dec' in cl or 'delta' in cl: col_map['dec'] = c
    if 'energy' in cl or 'energ' in cl: col_map['energy'] = c

print(f'Column mapping: {col_map}')

ra  = np.array(df[col_map.get('ra',  'RA')], dtype=float)
dec = np.array(df[col_map.get('dec', 'Dec')], dtype=float)
energy = np.array(df[col_map.get('energy', 'Energy')], dtype=float)

# Convert to galactic
coords = SkyCoord(ra=ra*u.deg, dec=dec*u.deg, frame='icrs')
gal    = coords.galactic
l_cr   = gal.l.deg
b_cr   = gal.b.deg

print(f'Events: {len(ra)}')
print(f'Energy range: {energy.min():.1f} – {energy.max():.1f} EeV')
print(f'RA range:     {ra.min():.1f}° – {ra.max():.1f}°')
print(f'Dec range:    {dec.min():.1f}° – {dec.max():.1f}°')

In [ ]:
# === AUGER EXPOSURE MODEL ===
# Auger exposure omega(dec) ∝ cos(lat)*cos(dec)*sin(alpha_M) + alpha_M*sin(lat)*sin(dec)
# where alpha_M depends on the maximum zenith angle (60°)
# Reference: Sommers 2001, Astroparticle Physics 14, 271

ZENITH_MAX = 60.0  # degrees
lat_rad = np.radians(AUGER_LAT)
theta_max_rad = np.radians(ZENITH_MAX)

def auger_exposure(dec_deg):
    """Auger relative exposure as function of declination."""
    dec_rad = np.radians(dec_deg)
    xi = (np.cos(theta_max_rad) - np.sin(lat_rad)*np.sin(dec_rad)) / \
         (np.cos(lat_rad)*np.cos(dec_rad))
    xi = np.clip(xi, -1, 1)
    # For dec where full circle is visible or not visible
    always_vis = (dec_rad - lat_rad) < -theta_max_rad  # always above horizon? No.
    never_vis  = (dec_rad - lat_rad) >  theta_max_rad
    alpha_M = np.arccos(xi)
    omega = np.where(
        never_vis, 0.0,
        np.cos(lat_rad)*np.cos(dec_rad)*np.sin(alpha_M) +
        alpha_M*np.sin(lat_rad)*np.sin(dec_rad)
    )
    return np.maximum(omega, 0.0)

# Test exposure model
dec_test = np.linspace(-90, 90, 181)
exp_test  = auger_exposure(dec_test)
print(f'Peak exposure at dec ≈ {dec_test[np.argmax(exp_test)]:.0f}°')
print(f'Exposure at KTF³ axis dec ({ktf3_coord.icrs.dec.deg:.1f}°): '
      f'{auger_exposure(ktf3_coord.icrs.dec.deg):.3f}')

# Compute exposure weights for each event
exp_weights = auger_exposure(dec)
exp_weights /= exp_weights.sum()  # normalize

In [ ]:
# === MEASURE DIPOLE FROM DATA ===
# Exposure-corrected dipole: weight each event by 1/omega(dec)

# Event unit vectors
ra_rad  = np.radians(ra)
dec_rad = np.radians(dec)
vx = np.cos(dec_rad)*np.cos(ra_rad)
vy = np.cos(dec_rad)*np.sin(ra_rad)
vz = np.sin(dec_rad)

# Inverse-exposure weights (avoid division by zero)
omega = auger_exposure(dec)
inv_omega = np.where(omega > 0.01, 1.0/omega, 0.0)
inv_omega /= inv_omega.sum()

# Weighted dipole
dx = np.sum(inv_omega * vx)
dy = np.sum(inv_omega * vy)
dz = np.sum(inv_omega * vz)
dipole_vec = np.array([dx, dy, dz])
dipole_vec /= np.linalg.norm(dipole_vec)

# Convert to galactic
dipole_icrs = SkyCoord(x=dipole_vec[0], y=dipole_vec[1], z=dipole_vec[2],
                       representation_type='cartesian', frame='icrs')
dipole_gal  = dipole_icrs.galactic
l_dip = dipole_gal.l.deg
b_dip = dipole_gal.b.deg

print(f'Measured CR dipole (exposure-corrected):')
print(f'  l = {l_dip:.1f}°, b = {b_dip:.1f}°')
print(f'Literature Auger dipole: l={AUGER_LIT_L}°, b={AUGER_LIT_B}°')

# Separation from KTF3 axis
dip_vec = gal_to_vec(l_dip, b_dip)
sep_meas = np.degrees(np.arccos(np.clip(np.dot(ktf3_vec, dip_vec), -1, 1)))
print(f'Separation KTF³ vs measured dipole: {sep_meas:.1f}°')
print(f'Separation KTF³ vs literature:      {sep_lit:.1f}°')

In [ ]:
# === MONTE CARLO NULL TEST ===
# Generate isotropic sky samples weighted by Auger exposure
# and measure the distribution of KTF3-dipole separations.

N_MC   = 5000
N_EVT  = len(ra)

# Precompute exposure map over dec grid
dec_grid = np.linspace(-90, 90, 1800)
exp_grid = auger_exposure(dec_grid)
exp_grid /= exp_grid.sum()

sep_mc = np.zeros(N_MC)
for i in range(N_MC):
    # Sample RA uniformly, Dec from exposure distribution
    ra_mc  = np.random.uniform(0, 360, N_EVT)
    dec_mc = np.random.choice(dec_grid, size=N_EVT, p=exp_grid)
    dec_mc += np.random.uniform(-0.05, 0.05, N_EVT)  # small jitter
    
    # Compute dipole
    ra_r = np.radians(ra_mc)
    de_r = np.radians(dec_mc)
    om   = auger_exposure(dec_mc)
    iom  = np.where(om > 0.01, 1.0/om, 0.0)
    iom /= iom.sum() if iom.sum() > 0 else 1
    dx = np.sum(iom * np.cos(de_r)*np.cos(ra_r))
    dy = np.sum(iom * np.cos(de_r)*np.sin(ra_r))
    dz = np.sum(iom * np.sin(de_r))
    dv = np.array([dx, dy, dz])
    norm = np.linalg.norm(dv)
    if norm > 0: dv /= norm
    sep_mc[i] = np.degrees(np.arccos(np.clip(np.dot(ktf3_vec, dv), -1, 1)))

# Sigma
mean_mc = sep_mc.mean()
std_mc  = sep_mc.std()
sigma_proj = (mean_mc - sep_meas) / std_mc  # positive = closer than expected
p_value    = np.mean(sep_mc <= sep_meas)

print(f'MC null: mean sep = {mean_mc:.1f}°, std = {std_mc:.1f}°')
print(f'Observed separation: {sep_meas:.1f}°')
print(f'σ = {sigma_proj:.3f}')
print(f'p = {p_value:.4f}')
print()
if abs(sigma_proj) < 2:
    print('Verdict: NULL — separation consistent with exposure-corrected random.')
else:
    print(f'Verdict: {sigma_proj:.1f}σ deviation from null — investigate further.')

In [ ]:
# === PLOTS ===
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('v52 — KTF³ vs Auger UHECR (Exposure-Corrected)', fontsize=13, fontweight='bold')

# Panel 1: Sky map
ax = axes[0]
ax.scatter(l_cr, b_cr, c=energy, cmap='plasma', s=20, alpha=0.7, zorder=3)
ax.scatter(KTF3_L, KTF3_B, c='red', s=150, marker='*',
           label=f'KTF³ axis (l={KTF3_L}°, b={KTF3_B}°)', zorder=5)
ax.scatter(l_dip, b_dip, c='lime', s=120, marker='D',
           label=f'Measured dipole (l={l_dip:.0f}°, b={b_dip:.0f}°)', zorder=5)
ax.scatter(AUGER_LIT_L, AUGER_LIT_B, c='orange', s=100, marker='^',
           label=f'Literature dipole (l={AUGER_LIT_L}°)', zorder=5)
ax.set_xlabel('Galactic longitude l (°)')
ax.set_ylabel('Galactic latitude b (°)')
ax.set_title(f'Auger CR Events (N={len(ra)})')
ax.legend(fontsize=7)
ax.grid(alpha=0.3)
ax.set_xlim(0, 360)
ax.set_ylim(-90, 90)

# Panel 2: Exposure model
ax = axes[1]
dec_plt = np.linspace(-90, 90, 500)
ax.plot(dec_plt, auger_exposure(dec_plt)/auger_exposure(dec_plt).max(),
        color='steelblue', lw=2, label='Exposure ω(δ)')
ax.axvline(AUGER_LAT, color='gray', ls='--', label=f'Auger lat={AUGER_LAT}°')
ax.axvline(ktf3_coord.icrs.dec.deg, color='red', ls=':', label='KTF³ dec')
ax.set_xlabel('Declination (°)')
ax.set_ylabel('Relative exposure')
ax.set_title('Auger Exposure Model (Sommers 2001)')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# Panel 3: MC null distribution
ax = axes[2]
ax.hist(sep_mc, bins=50, color='gray', edgecolor='white', alpha=0.8,
        density=True, label='MC null (exposure-corrected)')
ax.axvline(sep_meas, color='lime', lw=2.5,
           label=f'Measured sep = {sep_meas:.1f}°')
ax.axvline(sep_lit, color='orange', lw=2, ls='--',
           label=f'Literature sep = {sep_lit:.1f}°')
ax.set_xlabel('Separation from KTF³ axis (°)')
ax.set_ylabel('Density')
ax.set_title(f'MC Null: σ = {sigma_proj:.2f}')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('v52_auger_ktf3.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved: v52_auger_ktf3.png')

In [ ]:
# === RESULT SUMMARY ===
print('=' * 60)
print('v52 — KTF³ vs Auger UHECR Dipole (Exposure-Corrected)')
print('=' * 60)
print(f'Data:                 {"SYNTHETIC" if USE_SYNTHETIC else "Auger Open Data (auger_catalogSD.csv)"}')
print(f'N events:             {len(ra)}')
print(f'Energy threshold:     32 EeV')
print()
print(f'KTF³ axis:            l={KTF3_L}°, b={KTF3_B}°')
print(f'Measured CR dipole:   l={l_dip:.1f}°, b={b_dip:.1f}°')
print(f'Literature dipole:    l={AUGER_LIT_L}°, b={AUGER_LIT_B}°')
print()
print(f'Sep (measured):       {sep_meas:.1f}°')
print(f'Sep (literature):     {sep_lit:.1f}°')
print(f'MC null mean sep:     {mean_mc:.1f}° ± {std_mc:.1f}°')
print(f'σ (exposure-corr):    {sigma_proj:.3f}')
print(f'p-value:              {p_value:.4f}')
print()
print('VERDICT: NULL')
print('  Separation consistent with exposure-corrected isotropic null.')
print('  N=109 events insufficient for definitive test.')
print('  Geometric alignment ~22-23° noted but not significant.')
print()
print('HISTORY:')
print('  v52 (naive):           σ=6.31  ARTEFACT (no exposure correction)')
print('  v52b (corrected):      σ=0.42  NULL')
print('  v52 (this, clean):     σ={:.2f}  NULL'.format(sigma_proj))
print()
print('Next: Auger full dataset (TA collaboration) for N>1000 events.')
print('=' * 60)